# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Logistic Regression** </center>
---
**Profesor**: Pablo Camarillo Ramirez\
**Estudiante**: Sebastian Tadeo Quiroz Tejeda

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils
su = SparkUtils("ML: Logistic Regression", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/20 23:02:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Collect Data

In [2]:
from pcamarillor.spark_utils import SparkUtils
# Create a small dataset as a list of tuples
# Format: (label, feature_x1, feature_x2)
data = [
    (1.0, 2.0, 3.0),
    (0.0, 1.0, 2.5),
    (1.0, 3.0, 5.0),
    (0.0, 0.5, 1.0),
    (1.0, 4.0, 6.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("label", "float"), 
                                     ("feature_x1", "float"),
                                     ("feature_x2", "float")])

# Convert list to a DataFrame
df = su.spark.createDataFrame(data, schema)

### Assemble the features into a single vector column

In [3]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["feature_x1", "feature_x2"], outputCol="features")
data_with_features = assembler.transform(df).select("label", "features")
data_with_features.printSchema()                                   

root
 |-- label: float (nullable = true)
 |-- features: vector (nullable = true)



# Data splitting
#### 80% training data and 20% testing data

In [4]:
train_df, test_df = data_with_features.randomSplit([0.8, 0.2], seed=57)

### Show dataset (for debugging)

In [5]:
print("Original Dataset")
df.show()

# Print train dataset
print("train set")
train_df.show()

Original Dataset


+-----+----------+----------+
|label|feature_x1|feature_x2|
+-----+----------+----------+
|  1.0|       2.0|       3.0|
|  0.0|       1.0|       2.5|
|  1.0|       3.0|       5.0|
|  0.0|       0.5|       1.0|
|  1.0|       4.0|       6.0|
+-----+----------+----------+

train set


+-----+---------+
|label| features|
+-----+---------+
|  0.0|[1.0,2.5]|
|  1.0|[2.0,3.0]|
|  0.0|[0.5,1.0]|
|  1.0|[4.0,6.0]|
+-----+---------+



# Create ML Model

In [18]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(maxIter=10, regParam=0.01)

# Train ML Model

In [7]:
lr_model = lr.fit(train_df)

# Print coefficients
print("Coefficients: " + str(lr_model.coefficients))

# Display model summary
training_summary = lr_model.summary

26/04/16 01:16:27 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Coefficients: [2.346116998875653,0.7963873036415706]


## Predictions

In [8]:
# Use the trained model to make predictions on the test data
predictions = lr_model.transform(test_df)

# Show predictions
predictions.select("features", "prediction", "probability").show()

+---------+----------+--------------------+
| features|prediction|         probability|
+---------+----------+--------------------+
|[3.0,5.0]|       1.0|[0.00524886113385...|
+---------+----------+--------------------+



# Test ML Model

In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label",
                            predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, 
                  {evaluator.metricName: "accuracy"})
print(f"Accuracy: {accuracy}")
precision = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedPrecision"})
print(f"Precision: {precision}")
recall = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedRecall"})
print(f"Recall: {recall}")
f1 = evaluator.evaluate(predictions,
                {evaluator.metricName: "f1"})
print(f"F1 Score: {f1}")  

Accuracy: 0.8326848249027238
Precision: 0.8112390516267249
Recall: 0.8326848249027237
F1 Score: 0.7680827695624257


# Lab 10: Logistic regression to predict heart disease

# Data collection

In [6]:
# Define schema for the DataFrame
heart_schema = SparkUtils.generate_schema([
    ("male", "int"), 
    ("age", "int"), 
    ("education", "int"), 
    ("currentSmoker", "int"), 
    ("cigsPerDay", "int"), 
    ("BPMeds", "int"), 
    ("prevalentStroke", "int"), 
    ("prevalentHyp", "int"), 
    ("diabetes", "int"), 
    ("totChol", "int"), 
    ("sysBP", "float"), 
    ("diaBP", "float"), 
    ("BMI", "float"), 
    ("heartRate", "int"), 
    ("glucose", "int"), 
    ("TenYearCHD", "int")])

# Source: https://www.kaggle.com/datasets/dileep070/heart-disease-prediction-using-logistic-regression?resource=download

heart_df = su.spark.read \
                .option("header", "true") \
                .schema(heart_schema) \
                .csv("/opt/spark/work-dir/data/ml/logistic_regression/framingham.csv")

heart_df.printSchema()

root
 |-- male: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- education: integer (nullable = true)
 |-- currentSmoker: integer (nullable = true)
 |-- cigsPerDay: integer (nullable = true)
 |-- BPMeds: integer (nullable = true)
 |-- prevalentStroke: integer (nullable = true)
 |-- prevalentHyp: integer (nullable = true)
 |-- diabetes: integer (nullable = true)
 |-- totChol: integer (nullable = true)
 |-- sysBP: float (nullable = true)
 |-- diaBP: float (nullable = true)
 |-- BMI: float (nullable = true)
 |-- heartRate: integer (nullable = true)
 |-- glucose: integer (nullable = true)
 |-- TenYearCHD: integer (nullable = true)



In [ ]:
assembler = VectorAssembler(inputCols=["male",
                                        "age", 
                                        "currentSmoker", 
                                        "cigsPerDay", 
                                        "BPMeds", 
                                        "prevalentStroke", 
                                        "prevalentHyp", 
                                        "diabetes", 
                                        "totChol", 
                                        "sysBP", 
                                        "diaBP", 
                                        "BMI", 
                                        "heartRate", 
                                        "glucose"], 
                            outputCol="features",
                            handleInvalid="skip")
heart_df_featured = assembler.transform(heart_df).select("TenYearCHD", "features")
heart_df_featured = heart_df_featured.withColumnRenamed("TenYearCHD", "label")
heart_df_featured.printSchema()

root
 |-- label: integer (nullable = true)
 |-- features: vector (nullable = true)



# Data Splitting

In [22]:
train_df, test_df = heart_df_featured.randomSplit([0.8, 0.2], seed=57)

In [24]:
print("Original Dataset")
heart_df_featured.show(5)

print("Train set")
train_df.show()

Original Dataset
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[1,8,9,10,11,...|
|    0|[1.0,48.0,1.0,20....|
|    1|[0.0,61.0,1.0,30....|
|    0|[0.0,46.0,1.0,23....|
+-----+--------------------+
only showing top 5 rows
Train set
+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
|    0|(14,[0,1,8,9,10,1...|
+-----+--------------

# Create ML Model

In [25]:

lr = LogisticRegression(maxIter=10, regParam=0.01)


# Train ML Model

In [26]:
lr_model = lr.fit(train_df)

print(f"Coefficients: {lr_model.coefficients}")

training_summary = lr_model.summary

26/04/20 23:15:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Coefficients: [0.5679034392271173,0.05470664686870225,0.06852297850313113,0.01599270348085958,0.2096323533176199,0.751524038057282,0.2394234497981218,-0.0565235995218806,0.0019296903667571123,0.012486366151868447,0.0001583250611412812,-0.009318088589876934,-0.0006816070205812322,0.00723616661001617]


In [27]:
# Use the trained model to make predictions on the test data
predictions = lr_model.transform(test_df)

# Show predictions
predictions.select("features", "prediction", "probability").show()

+--------------------+----------+--------------------+
|            features|prediction|         probability|
+--------------------+----------+--------------------+
|(14,[0,1,8,9,10,1...|       0.0|[0.93107568129387...|
|(14,[0,1,8,9,10,1...|       0.0|[0.94846119629024...|
|(14,[0,1,8,9,10,1...|       0.0|[0.93763619585032...|
|(14,[0,1,8,9,10,1...|       0.0|[0.93620343842092...|
|(14,[0,1,8,9,10,1...|       0.0|[0.94063183092132...|
|(14,[0,1,8,9,10,1...|       0.0|[0.94098164731348...|
|(14,[0,1,8,9,10,1...|       0.0|[0.94057860846505...|
|(14,[0,1,8,9,10,1...|       0.0|[0.94228528336652...|
|(14,[0,1,8,9,10,1...|       0.0|[0.92604753850266...|
|(14,[0,1,8,9,10,1...|       0.0|[0.92372088779853...|
|(14,[0,1,8,9,10,1...|       0.0|[0.90988774011547...|
|(14,[0,1,8,9,10,1...|       0.0|[0.91940406309496...|
|(14,[0,1,8,9,10,1...|       0.0|[0.90518291480769...|
|(14,[0,1,8,9,10,1...|       0.0|[0.91282281525958...|
|(14,[0,1,8,9,10,1...|       0.0|[0.90153100850296...|
|(14,[0,1,

# Test ML Model

In [30]:
evaluator = MulticlassClassificationEvaluator(labelCol="label",
                            predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, 
                  {evaluator.metricName: "accuracy"})
print(f"Accuracy: {accuracy}")
precision = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedPrecision"})
print(f"Precision: {precision}")
recall = evaluator.evaluate(predictions,
                  {evaluator.metricName: "weightedRecall"})
print(f"Recall: {recall}")
f1 = evaluator.evaluate(predictions,
                {evaluator.metricName: "f1"})
print(f"F1 Score: {f1}")  

Accuracy: 0.8326848249027238
Precision: 0.8112390516267249
Recall: 0.8326848249027237
F1 Score: 0.7680827695624257


In [31]:
su.spark.stop()